In [ ]:
# =============================================
# STEP 5A: Mount Google Drive
# =============================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
# ============================================================
# ONE-CELL END-TO-END FAST RAGNET PIPELINE FOR NYC CRASH DATA
# FULL CORRECTED VERSION
#
# THIS VERSION:
# - loads raw NYC crash CSV file(s)
# - preprocesses and standardizes columns
# - builds a crash text field
# - applies the same corrected RAGNet pipeline/configuration style
#
# INCLUDED FIXES
# - same categorical class of same attribute -> same input embedding
# - shared event-based memory bank storing only training row indices
# - selective balanced memory construction
# - original masked value + original PLM embedding are removed
# - masked attributes are initialized from memory via theory-based retrieval
# - project_batch does NOT zero masked attributes
# - masked categorical F1 weighted by evaluated masked count
# - masked numerical MAE/MSE weighted by evaluated masked count
# ============================================================

# -----------------------------
# 0) INSTALLS
# -----------------------------
!pip -q install transformers==4.38.2 nltk scikit-learn pandas numpy tqdm

# -----------------------------
# 1) IMPORTS
# -----------------------------
import os, re, json, math, glob, random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except:
    pass

from transformers import AutoTokenizer, AutoModel
from google.colab import drive

# -----------------------------
# 2) CONFIG
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# You can set this to:
# - a single raw CSV file path
# - or a directory containing raw CSV files
RAW_DATA_PATH = "/content/drive/MyDrive/NYC_crash"

SAVE_DIR = "/content/drive/MyDrive/NYC_crash/ragnet_ultra_fast"
os.makedirs(SAVE_DIR, exist_ok=True)

OUT_DIR = os.path.join(SAVE_DIR, "outputs")
MODEL_DIR = os.path.join(SAVE_DIR, "models")
EMB_DIR = os.path.join(SAVE_DIR, "embeddings")
for p in [OUT_DIR, MODEL_DIR, EMB_DIR]:
    os.makedirs(p, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# PREPROCESSED FEATURE SET FOR NYC CRASH DATA
TEXTUAL_COLUMNS = ["crash_text"]
CATEGORICAL_COLUMNS = ["borough", "on_street_name", "contributing_factor_vehicle_1"]
NUMERICAL_COLUMNS = ["latitude", "longitude", "number_of_persons_injured", "number_of_persons_killed", "hour_of_day"]

# pseudo target used only for stratified split / balanced memory
TARGET_COLUMN = "borough"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

BATCH_SIZE_EMB = 128
TRAIN_BATCH_SIZE = 256
MAX_LEN_SUMMARY = 32
MAX_LEN_ATTR = 16

TRAIN_SUBSET = 1000000
TEST_SUBSET = 100000

# for first run keep modest, then increase
EPOCHS = 100
LR = 1e-4

TOP_K_EDGE = 2
MEMORY_LIMIT = 300

# training masking
USE_RANDOM_MASK_TRAINING = True
MASK_ONE_ATTR_PER_SAMPLE = True
MASK_PROB = 0.15

# IMPORTANT: loss on visible attrs only
LOSS_ON_MASKED_ONLY = False

# MARIP + regularization
USE_MARIP = True
MARIP_BLEND = 0.1
REG_WEIGHT = 1e-4

# same categorical class => same input embedding
USE_CLASS_LOOKUP_EMB = True

# same-class hidden consistency
USE_CAT_CLASS_CONSISTENCY_LOSS = True
CAT_CLASS_CONSISTENCY_WEIGHT = 1e-2
CONSISTENCY_ON_VISIBLE_ONLY = True

# memory-based missing init
USE_MISSING_INIT = True
MISSING_INIT_TOPK = 5
MISSING_INIT_R = 2

# balanced memory selection
MEMORY_NUM_BINS = 5

# raw data preprocessing controls
MAX_ROWS_AFTER_PREPROCESS = 120000   # cap for speed; set None to keep all
MIN_TEXT_LENGTH = 3

# -----------------------------
# 3) MOUNT DRIVE
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 4) LOAD RAW NYC CRASH FILE(S)
# -----------------------------
def discover_csv_files(path_like):
    if os.path.isfile(path_like):
        return [path_like]
    if os.path.isdir(path_like):
        files = sorted(glob.glob(os.path.join(path_like, "*.csv")))
        if len(files) == 0:
            raise ValueError(f"No CSV files found in directory: {path_like}")
        return files
    raise ValueError(f"RAW_DATA_PATH does not exist: {path_like}")

def load_raw_crash_files(path_like):
    csv_files = discover_csv_files(path_like)
    print("Found CSV files:", csv_files)
    dfs = []
    for fp in csv_files:
        try:
            tmp = pd.read_csv(fp, low_memory=False)
            tmp["__source_file__"] = os.path.basename(fp)
            dfs.append(tmp)
            print(f"Loaded {fp}: shape={tmp.shape}")
        except Exception as e:
            print(f"Skipping {fp} due to read error: {e}")
    if len(dfs) == 0:
        raise ValueError("No readable crash CSV files were loaded.")
    return pd.concat(dfs, ignore_index=True)

raw_df = load_raw_crash_files(RAW_DATA_PATH)
print("Combined raw shape:", raw_df.shape)

# -----------------------------
# 5) PREPROCESS RAW NYC CRASH DATA
# -----------------------------
def canonicalize_columns(df):
    out = df.copy()
    out.columns = [
        re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
        for c in out.columns
    ]
    return out

def parse_numeric_series(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = (
        series.astype(str)
        .str.replace(r"[^0-9\.\-]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")

def first_existing_column(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Could not find any of required columns: {candidates}")
    return None

def safe_text(x):
    if pd.isna(x):
        return "missing"
    x = str(x).strip()
    return x if x != "" else "missing"

def preprocess_nyc_crash_raw(raw_df):
    df = canonicalize_columns(raw_df)

    # resolve important raw columns
    col_borough = first_existing_column(df, ["borough"], required=False)
    col_on_street = first_existing_column(df, ["on_street_name"], required=False)
    col_cross_street = first_existing_column(df, ["cross_street_name"], required=False)
    col_off_street = first_existing_column(df, ["off_street_name"], required=False)

    col_factor1 = first_existing_column(
        df,
        ["contributing_factor_vehicle_1", "contributing_factor_1"],
        required=False
    )
    col_vehicle1 = first_existing_column(
        df,
        ["vehicle_type_code_1", "vehicle_type_1"],
        required=False
    )

    col_lat = first_existing_column(df, ["latitude"], required=False)
    col_lon = first_existing_column(df, ["longitude"], required=False)

    col_persons_inj = first_existing_column(df, ["number_of_persons_injured"], required=False)
    col_persons_killed = first_existing_column(df, ["number_of_persons_killed"], required=False)

    col_crash_date = first_existing_column(df, ["crash_date"], required=False)
    col_crash_time = first_existing_column(df, ["crash_time"], required=False)

    col_collision_id = first_existing_column(df, ["collision_id"], required=False)

    # fill missing fallback columns if absent
    if col_borough is None:
        df["borough"] = "Missing"
        col_borough = "borough"
    if col_on_street is None:
        df["on_street_name"] = "Missing"
        col_on_street = "on_street_name"
    if col_factor1 is None:
        df["contributing_factor_vehicle_1"] = "Missing"
        col_factor1 = "contributing_factor_vehicle_1"
    if col_vehicle1 is None:
        df["vehicle_type_code_1"] = "Missing"
        col_vehicle1 = "vehicle_type_code_1"
    if col_lat is None:
        df["latitude"] = np.nan
        col_lat = "latitude"
    if col_lon is None:
        df["longitude"] = np.nan
        col_lon = "longitude"
    if col_persons_inj is None:
        df["number_of_persons_injured"] = np.nan
        col_persons_inj = "number_of_persons_injured"
    if col_persons_killed is None:
        df["number_of_persons_killed"] = np.nan
        col_persons_killed = "number_of_persons_killed"

    # normalize text columns
    for c in [col_borough, col_on_street, col_cross_street, col_off_street, col_factor1, col_vehicle1]:
        if c is not None:
            df[c] = df[c].fillna("Missing").astype(str).str.strip()
            df.loc[df[c] == "", c] = "Missing"

    # numeric conversions
    for c in [col_lat, col_lon, col_persons_inj, col_persons_killed]:
        df[c] = parse_numeric_series(df[c])

    # parse date/time
    if col_crash_date is not None:
        df["_parsed_crash_date"] = pd.to_datetime(df[col_crash_date], errors="coerce")
    else:
        df["_parsed_crash_date"] = pd.NaT

    def parse_hour_from_time(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip()
        m = re.match(r"^(\d{1,2}):(\d{2})", x)
        if m:
            hh = int(m.group(1))
            if 0 <= hh <= 23:
                return float(hh)
        return np.nan

    if col_crash_time is not None:
        df["hour_of_day"] = df[col_crash_time].apply(parse_hour_from_time)
    else:
        df["hour_of_day"] = np.nan

    # canonical feature names expected downstream
    df["borough"] = df[col_borough].copy()
    df["on_street_name"] = df[col_on_street].copy()
    df["contributing_factor_vehicle_1"] = df[col_factor1].copy()
    df["vehicle_type_code_1"] = df[col_vehicle1].copy()
    df["latitude"] = df[col_lat].copy()
    df["longitude"] = df[col_lon].copy()
    df["number_of_persons_injured"] = df[col_persons_inj].copy()
    df["number_of_persons_killed"] = df[col_persons_killed].copy()

    # create crash text
    def build_crash_text(row):
        borough = safe_text(row["borough"]).lower()
        street = safe_text(row["on_street_name"]).lower()

        cross_street = safe_text(row[col_cross_street]).lower() if col_cross_street is not None else "missing"
        off_street = safe_text(row[col_off_street]).lower() if col_off_street is not None else "missing"

        factor = safe_text(row["contributing_factor_vehicle_1"]).lower()
        vehicle = safe_text(row["vehicle_type_code_1"]).lower()

        if pd.notna(row["_parsed_crash_date"]):
            weekday = row["_parsed_crash_date"].day_name().lower()
            month = row["_parsed_crash_date"].month_name().lower()
        else:
            weekday = "unknown_day"
            month = "unknown_month"

        if pd.notna(row["hour_of_day"]):
            hh = int(row["hour_of_day"])
            if hh < 6:
                time_bucket = "night"
            elif hh < 12:
                time_bucket = "morning"
            elif hh < 18:
                time_bucket = "afternoon"
            else:
                time_bucket = "evening"
        else:
            time_bucket = "unknown_time"

        injured = row["number_of_persons_injured"]
        killed = row["number_of_persons_killed"]

        injured_phrase = "unknown injuries" if pd.isna(injured) else f"{int(injured)} injured"
        killed_phrase = "unknown fatalities" if pd.isna(killed) else f"{int(killed)} killed"

        location_phrase = street
        if cross_street != "missing":
            location_phrase = f"{street} near {cross_street}"
        elif off_street != "missing":
            location_phrase = f"{street} near {off_street}"

        return (
            f"Crash in {borough} on {location_phrase} during {time_bucket} "
            f"on {weekday} in {month}. Factor {factor}. Vehicle {vehicle}. "
            f"{injured_phrase}, {killed_phrase}."
        )

    df["crash_text"] = df.apply(build_crash_text, axis=1)

    # final feature subset
    keep_cols = TEXTUAL_COLUMNS + CATEGORICAL_COLUMNS + NUMERICAL_COLUMNS
    out = df[keep_cols].copy()

    # cleanup text / category
    for col in TEXTUAL_COLUMNS + CATEGORICAL_COLUMNS:
        out[col] = out[col].fillna("Missing").astype(str).str.strip()
        out.loc[out[col] == "", col] = "Missing"

    # numeric cleanup
    for col in NUMERICAL_COLUMNS:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # optional geographic sanity filters
    if "latitude" in out.columns:
        out.loc[(out["latitude"] < -90) | (out["latitude"] > 90), "latitude"] = np.nan
    if "longitude" in out.columns:
        out.loc[(out["longitude"] < -180) | (out["longitude"] > 180), "longitude"] = np.nan

    # drop rows that are completely useless
    non_empty_mask = (
        out[TEXTUAL_COLUMNS].apply(lambda s: s.astype(str).str.len() >= MIN_TEXT_LENGTH).any(axis=1)
        | out[CATEGORICAL_COLUMNS].ne("Missing").any(axis=1)
        | out[NUMERICAL_COLUMNS].notna().any(axis=1)
    )
    out = out.loc[non_empty_mask].reset_index(drop=True)

    # cap size for speed
    if MAX_ROWS_AFTER_PREPROCESS is not None and len(out) > MAX_ROWS_AFTER_PREPROCESS:
        out = out.sample(MAX_ROWS_AFTER_PREPROCESS, random_state=SEED).reset_index(drop=True)

    return out

df = preprocess_nyc_crash_raw(raw_df)
print("Preprocessed shape:", df.shape)
print(df.head(3))

prepared_csv = os.path.join(SAVE_DIR, "nyc_crash_prepared.csv")
df.to_csv(prepared_csv, index=False)

# -----------------------------
# 6) ATTRIBUTE TEXTS
# -----------------------------
def make_attr_text(name, value):
    if pd.isna(value):
        return f"{name}: missing"
    return f"{name}: {value}"

for col in CATEGORICAL_COLUMNS:
    df[f"{col}_attr_text"] = df[col].apply(lambda x: make_attr_text(col, x))
for col in NUMERICAL_COLUMNS:
    df[f"{col}_attr_text"] = df[col].apply(lambda x: make_attr_text(col, x))

# -----------------------------
# 7) ENCODE CATEGORICAL
# -----------------------------
cat_maps = {}
cat_encoded_df = pd.DataFrame()
num_classes_per_cat_attr = []

for col in CATEGORICAL_COLUMNS:
    le = LabelEncoder()
    enc = le.fit_transform(df[col].astype(str))
    cat_encoded_df[col] = enc
    cat_maps[col] = [str(x) for x in le.classes_]
    num_classes_per_cat_attr.append(len(le.classes_))

target_le = LabelEncoder()
target_labels = target_le.fit_transform(df[TARGET_COLUMN].astype(str))
target_map = {int(i): str(v) for i, v in enumerate(target_le.classes_)}

# -----------------------------
# 8) NUMERICAL VALUES
# -----------------------------
num_values_df = df[NUMERICAL_COLUMNS].copy()
num_values_raw = num_values_df.fillna(-1).to_numpy(dtype=np.float32)

scaled_num = np.zeros_like(num_values_raw, dtype=np.float32)
num_scale_stats = {}
for j in range(num_values_raw.shape[1]):
    col = num_values_raw[:, j]
    mask = col != -1
    if mask.sum() > 0:
        mean_j = col[mask].mean()
        std_j = col[mask].std()
        if std_j < 1e-8:
            std_j = 1.0
        scaled_num[mask, j] = (col[mask] - mean_j) / std_j
        scaled_num[~mask, j] = -1.0
        num_scale_stats[NUMERICAL_COLUMNS[j]] = {"mean": float(mean_j), "std": float(std_j)}
    else:
        scaled_num[:, j] = -1.0
        num_scale_stats[NUMERICAL_COLUMNS[j]] = {"mean": 0.0, "std": 1.0}

# -----------------------------
# 9) LOAD EMBEDDING MODEL
# -----------------------------
emb_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
emb_model = AutoModel.from_pretrained(EMBED_MODEL_NAME).to(DEVICE)
if DEVICE.type == "cuda":
    emb_model = emb_model.half()
emb_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

def embed_text_list(text_list, batch_size=128, max_length=16):
    all_embs = []
    for i in tqdm(range(0, len(text_list), batch_size), leave=False):
        batch = text_list[i:i+batch_size]
        encoded = emb_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                out = emb_model(**encoded)
                emb = mean_pooling(out, encoded["attention_mask"]).float()
        all_embs.append(emb.cpu().numpy())
        del encoded, out, emb
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return np.vstack(all_embs)

# -----------------------------
# 10) COMPUTE EMBEDDINGS
# -----------------------------
textual_embeddings = np.stack([
    embed_text_list(df[col].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_SUMMARY)
    for col in TEXTUAL_COLUMNS
], axis=1).astype(np.float32)

def build_same_class_categorical_embeddings(df, categorical_columns, batch_size=128, max_length=12):
    per_attr_embeddings = []
    categorical_class_embedding_maps = {}

    for col in categorical_columns:
        values = df[col].fillna("Missing").astype(str).str.strip().tolist()
        unique_vals = sorted(list(set(values)))
        unique_texts = [make_attr_text(col, v) for v in unique_vals]
        unique_embs = embed_text_list(unique_texts, batch_size=batch_size, max_length=max_length).astype(np.float32)
        val_to_emb = {v: unique_embs[i] for i, v in enumerate(unique_vals)}
        categorical_class_embedding_maps[col] = val_to_emb
        row_embs = np.stack([val_to_emb[v] for v in values], axis=0).astype(np.float32)
        per_attr_embeddings.append(row_embs)

    return np.stack(per_attr_embeddings, axis=1).astype(np.float32), categorical_class_embedding_maps

if USE_CLASS_LOOKUP_EMB:
    categorical_embeddings, categorical_class_embedding_maps = build_same_class_categorical_embeddings(
        df=df,
        categorical_columns=CATEGORICAL_COLUMNS,
        batch_size=BATCH_SIZE_EMB,
        max_length=MAX_LEN_ATTR
    )
else:
    categorical_embeddings = np.stack([
        embed_text_list(df[f"{col}_attr_text"].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_ATTR)
        for col in CATEGORICAL_COLUMNS
    ], axis=1).astype(np.float32)
    categorical_class_embedding_maps = None

numerical_embeddings = np.stack([
    embed_text_list(df[f"{col}_attr_text"].fillna("").astype(str).tolist(), BATCH_SIZE_EMB, MAX_LEN_ATTR)
    for col in NUMERICAL_COLUMNS
], axis=1).astype(np.float32)

all_embeddings_np = np.concatenate([textual_embeddings, categorical_embeddings, numerical_embeddings], axis=1)
all_embeddings = torch.tensor(all_embeddings_np, dtype=torch.float32)

normalized_embeddings = all_embeddings.clone()
N, M, D = normalized_embeddings.shape
for m in range(M):
    attr = all_embeddings[:, m, :]
    nz = (attr.abs().sum(dim=1) != 0)
    if nz.sum() > 0:
        mean_m = attr[nz].mean(dim=0)
        std_m = attr[nz].std(dim=0)
        normalized_embeddings[nz, m, :] = (attr[nz] - mean_m) / (std_m + 1e-8)

# -----------------------------
# 11) BUILD GROUND TRUTH
# -----------------------------
ground_truth_text = df[TEXTUAL_COLUMNS].fillna("").astype(str).to_numpy()
ground_truth_cat = cat_encoded_df.to_numpy(dtype=np.int64)
ground_truth_num = scaled_num.astype(np.float32)

MODALITIES = ["txt"] * len(TEXTUAL_COLUMNS) + ["cat"] * len(CATEGORICAL_COLUMNS) + ["num"] * len(NUMERICAL_COLUMNS)
ALL_ATTRIBUTE_NAMES = TEXTUAL_COLUMNS + CATEGORICAL_COLUMNS + NUMERICAL_COLUMNS

categorical_indices = cat_encoded_df.to_numpy(dtype=np.int64)
numerical_values = num_values_raw.astype(np.float32)

categorical_indices_tensor = torch.tensor(categorical_indices, dtype=torch.long)
numerical_values_tensor = torch.tensor(numerical_values, dtype=torch.float32)

# -----------------------------
# 12) TRAIN/TEST SPLIT
# -----------------------------
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=target_labels
)

train_idx = train_idx[:min(TRAIN_SUBSET, len(train_idx))]
test_idx = test_idx[:min(TEST_SUBSET, len(test_idx))]
test_df = df.iloc[test_idx].reset_index(drop=True)

# -----------------------------
# 13) HELPERS
# -----------------------------
def safe_word_tokenize(x):
    try:
        return nltk.word_tokenize(str(x))
    except:
        return str(x).split()

def compute_text_bleu(text_refs, text_preds):
    smoothie = SmoothingFunction().method1
    scores = []
    for ref, pred in zip(text_refs, text_preds):
        ref_tokens = safe_word_tokenize(ref)
        pred_tokens = safe_word_tokenize(pred)
        if len(ref_tokens) == 0 or len(pred_tokens) == 0:
            scores.append(0.0)
            continue
        scores.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie))
    return float(np.mean(scores)) if len(scores) else 0.0

def compute_weighted_cat_f1_per_attribute(cat_true, cat_pred, categorical_columns):
    results = {}
    scores = []
    for j, col in enumerate(categorical_columns):
        f1 = f1_score(cat_true[:, j], cat_pred[:, j], average="weighted", zero_division=0)
        results[col] = float(f1)
        scores.append(f1)
    results["weighted_attribute_f1"] = float(np.mean(scores)) if len(scores) else 0.0
    return results

def compute_weighted_num_mae_per_attribute(num_true, num_pred, numerical_columns):
    results = {}
    maes = []
    for j, col in enumerate(numerical_columns):
        valid = num_true[:, j] != -1
        if valid.any():
            mae = np.mean(np.abs(num_true[valid, j] - num_pred[valid, j]))
            results[col] = float(mae)
            maes.append(mae)
        else:
            results[col] = None
    results["weighted_attribute_mae"] = float(np.mean(maes)) if len(maes) else 0.0
    return results

def build_training_attr_mask(batch_size, num_attrs, device, mask_one_attr_per_sample=True, mask_prob=0.15):
    attr_mask = torch.ones(batch_size, num_attrs, device=device)
    if mask_one_attr_per_sample:
        masked_attr = torch.randint(0, num_attrs, (batch_size,), device=device)
        attr_mask[torch.arange(batch_size, device=device), masked_attr] = 0.0
    else:
        rand_mask = (torch.rand(batch_size, num_attrs, device=device) < mask_prob).float()
        empty_rows = rand_mask.sum(dim=1) == 0
        if empty_rows.any():
            row_ids = torch.where(empty_rows)[0]
            forced = torch.randint(0, num_attrs, (row_ids.numel(),), device=device)
            rand_mask[row_ids] = 0.0
            rand_mask[row_ids, forced] = 1.0
        attr_mask = 1.0 - rand_mask
    return attr_mask

def apply_input_mask(llm, rawn, cati, attr_mask):
    llm = llm.clone()
    rawn = rawn.clone()
    cati = cati.clone()

    for attr_id, mod in enumerate(MODALITIES):
        masked_rows = attr_mask[:, attr_id] == 0
        if not masked_rows.any():
            continue

        llm[masked_rows, attr_id, :] = 0.0

        if mod == "cat":
            local_cat = attr_id - len(TEXTUAL_COLUMNS)
            cati[masked_rows, local_cat] = 0

        elif mod == "num":
            local_num = attr_id - len(TEXTUAL_COLUMNS) - len(CATEGORICAL_COLUMNS)
            rawn[masked_rows, local_num] = 0.0

    return llm, rawn, cati

def hidden_regularization(H, attr_mask=None):
    reg = (H ** 2).mean()
    if attr_mask is not None:
        masked_h = H * (1.0 - attr_mask.unsqueeze(-1))
        reg = reg + 0.1 * (masked_h ** 2).mean()
    return reg

def categorical_class_consistency_loss(H, cat_targets, attr_mask=None, visible_only=True):
    total_loss = 0.0
    total_terms = 0

    for j in range(len(CATEGORICAL_COLUMNS)):
        attr_id = len(TEXTUAL_COLUMNS) + j
        h = H[:, attr_id, :]
        y = cat_targets[:, j]

        if attr_mask is not None and visible_only:
            valid_rows = (attr_mask[:, attr_id] == 1)
        else:
            valid_rows = torch.ones(h.size(0), dtype=torch.bool, device=h.device)

        if valid_rows.sum() < 2:
            continue

        y_valid = y[valid_rows]
        h_valid = h[valid_rows]

        unique_classes = torch.unique(y_valid)
        for cls in unique_classes:
            cls_rows = (y_valid == cls)
            if cls_rows.sum() < 2:
                continue

            h_cls = h_valid[cls_rows]
            proto = h_cls.mean(dim=0, keepdim=True)
            total_loss = total_loss + ((h_cls - proto) ** 2).mean()
            total_terms += 1

    if total_terms == 0:
        return torch.tensor(0.0, device=H.device)

    return total_loss / total_terms

def weighted_average_by_count(items, value_key, count_key):
    valid_items = [
        x for x in items
        if x.get(value_key) is not None and x.get(count_key) is not None
    ]
    total_weight = sum(x[count_key] for x in valid_items)
    if total_weight <= 0:
        return None, 0
    score = sum(x[value_key] * x[count_key] for x in valid_items) / total_weight
    return float(score), int(total_weight)

def build_balanced_memory_row_ids(
    train_idx,
    df,
    categorical_columns,
    numerical_columns,
    target_column,
    memory_limit=300,
    bins_per_num=5,
    seed=42,
):
    rng = np.random.RandomState(seed)
    train_idx = np.array(train_idx, dtype=int)
    train_df = df.iloc[train_idx].copy()

    selected = []
    selected_set = set()

    def add_row(rid):
        rid = int(rid)
        if rid not in selected_set and len(selected) < memory_limit:
            selected.append(rid)
            selected_set.add(rid)

    if target_column in train_df.columns:
        for _, sub in train_df.groupby(target_column):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    for col in categorical_columns:
        for _, sub in train_df.groupby(col):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    for col in numerical_columns:
        valid_mask = train_df[col].notna()
        valid_df = train_df.loc[valid_mask, [col]]
        if len(valid_df) == 0:
            continue

        try:
            n_bins = min(bins_per_num, max(1, valid_df[col].nunique()))
            bin_ids = pd.qcut(valid_df[col], q=n_bins, duplicates="drop")
        except:
            continue

        tmp = valid_df.copy()
        tmp["_bin"] = bin_ids

        for _, sub in tmp.groupby("_bin"):
            candidate_rows = sub.index.to_numpy()
            rng.shuffle(candidate_rows)
            if len(candidate_rows) > 0:
                add_row(candidate_rows[0])

    if len(selected) < memory_limit:
        for col in categorical_columns:
            value_counts = train_df[col].value_counts()
            rare_first = value_counts.sort_values().index.tolist()

            for val in rare_first:
                sub = train_df[train_df[col] == val]
                candidate_rows = sub.index.to_numpy()
                rng.shuffle(candidate_rows)
                for rid in candidate_rows[:2]:
                    add_row(rid)
                    if len(selected) >= memory_limit:
                        break
                if len(selected) >= memory_limit:
                    break
            if len(selected) >= memory_limit:
                break

    if len(selected) < memory_limit:
        remaining = [int(r) for r in train_idx if int(r) not in selected_set]
        rng.shuffle(remaining)
        for rid in remaining:
            add_row(rid)
            if len(selected) >= memory_limit:
                break

    return selected

# -----------------------------
# 14) EVENT-BASED MEMORY BANK
# -----------------------------
class EventMemoryBank:
    def __init__(self, normalized_embeddings, df, categorical_indices, numerical_values,
                 textual_columns, categorical_columns, numerical_columns, modalities):
        self.row_ids = []
        self.normalized_embeddings = normalized_embeddings
        self.df = df
        self.categorical_indices = categorical_indices
        self.numerical_values = numerical_values
        self.textual_columns = textual_columns
        self.categorical_columns = categorical_columns
        self.numerical_columns = numerical_columns
        self.modalities = modalities
        self.emb_cache = {}
        self.values_cache = {}
        self.valid_mask_cache = {}

    def set_row_ids(self, row_ids):
        self.row_ids = [int(x) for x in row_ids]
        self.emb_cache = {}
        self.values_cache = {}
        self.valid_mask_cache = {}

    def size(self):
        return len(self.row_ids)

    def get_row_ids(self):
        return self.row_ids

    def get_embeddings_for_attr(self, attr_id):
        if attr_id in self.emb_cache:
            return self.emb_cache[attr_id]
        if len(self.row_ids) == 0:
            return None
        emb = self.normalized_embeddings[self.row_ids, attr_id, :].detach().cpu()
        self.emb_cache[attr_id] = emb
        return emb

    def get_values_for_attr(self, attr_id):
        if attr_id in self.values_cache:
            return self.values_cache[attr_id]

        vals = []
        if self.modalities[attr_id] == "txt":
            local_txt = attr_id
            col = self.textual_columns[local_txt]
            for rid in self.row_ids:
                vals.append(str(self.df.iloc[rid][col]).strip())

        elif self.modalities[attr_id] == "cat":
            local_cat = attr_id - len(self.textual_columns)
            for rid in self.row_ids:
                vals.append(int(self.categorical_indices[rid, local_cat]))

        else:
            local_num = attr_id - len(self.textual_columns) - len(self.categorical_columns)
            for rid in self.row_ids:
                vals.append(float(self.numerical_values[rid, local_num]))

        self.values_cache[attr_id] = vals
        return vals

    def get_valid_mask_for_attr(self, attr_id):
        if attr_id in self.valid_mask_cache:
            return self.valid_mask_cache[attr_id]

        if len(self.row_ids) == 0:
            out = torch.zeros(0, dtype=torch.bool)
            self.valid_mask_cache[attr_id] = out
            return out

        mod = self.modalities[attr_id]

        if mod == "txt":
            local_txt = attr_id
            col = self.textual_columns[local_txt]
            mask = []
            for rid in self.row_ids:
                v = str(self.df.iloc[rid][col]).strip()
                mask.append(v != "")
            out = torch.tensor(mask, dtype=torch.bool)

        elif mod == "cat":
            out = torch.ones(len(self.row_ids), dtype=torch.bool)

        else:
            local_num = attr_id - len(self.textual_columns) - len(self.categorical_columns)
            mask = []
            for rid in self.row_ids:
                v = float(self.numerical_values[rid, local_num])
                mask.append(v != -1)
            out = torch.tensor(mask, dtype=torch.bool)

        self.valid_mask_cache[attr_id] = out
        return out

memory_bank = EventMemoryBank(
    normalized_embeddings=normalized_embeddings,
    df=df,
    categorical_indices=categorical_indices,
    numerical_values=numerical_values,
    textual_columns=TEXTUAL_COLUMNS,
    categorical_columns=CATEGORICAL_COLUMNS,
    numerical_columns=NUMERICAL_COLUMNS,
    modalities=MODALITIES
)

memory_row_ids = build_balanced_memory_row_ids(
    train_idx=train_idx,
    df=df,
    categorical_columns=CATEGORICAL_COLUMNS,
    numerical_columns=NUMERICAL_COLUMNS,
    target_column=TARGET_COLUMN,
    memory_limit=MEMORY_LIMIT,
    bins_per_num=MEMORY_NUM_BINS,
    seed=SEED,
)
memory_bank.set_row_ids(memory_row_ids)
print("Event-based memory bank built. size =", memory_bank.size())
print("First 10 memory row ids:", memory_row_ids[:10])

# -----------------------------
# 15) EDGE INDEX
# -----------------------------
def build_modality_based_edge_index(norm_embs, modality_types, top_k=2):
    N0, M0, D0 = norm_embs.shape
    attr_means = []
    for m in range(M0):
        e = norm_embs[:, m, :].detach().cpu()
        nz = (e.abs().sum(dim=1) != 0)
        if nz.any():
            mean_e = e[nz].mean(dim=0)
        else:
            mean_e = torch.zeros(D0)
        attr_means.append(mean_e)

    attr_means = torch.stack(attr_means, dim=0)
    attr_means = F.normalize(attr_means, dim=1)

    groups = defaultdict(list)
    for i, mod in enumerate(modality_types):
        groups[mod].append(i)

    edges = []
    for _, idxs in groups.items():
        if len(idxs) < 2:
            continue
        em = attr_means[idxs]
        sim = torch.matmul(em, em.T)
        for i, src in enumerate(idxs):
            top = torch.topk(sim[i], k=min(top_k + 1, len(idxs))).indices.tolist()
            for t in top:
                tgt = idxs[t]
                if src != tgt:
                    edges.append((src, tgt))
    edges += [(j, i) for (i, j) in edges]
    edges = list(set(edges))
    if len(edges) == 0:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(edges, dtype=torch.long).T

edge_index = build_modality_based_edge_index(normalized_embeddings, MODALITIES, top_k=TOP_K_EDGE).to(DEVICE)
print("edge_index:", edge_index.shape)

# -----------------------------
# 16) THEORY-BASED MISSING INIT
# -----------------------------
def initialize_missing_embeddings_theory(
    missing_attr_idx,
    known_attr,
    known_embedding,
    memory_bank,
    modality_types,
    k=5,
    r=2,
    d_h=384,
):
    device = known_embedding.device
    missing_modality = modality_types[missing_attr_idx]

    if known_embedding.dim() != 2:
        raise ValueError("known_embedding must be [num_attributes, d_h] for one sample")

    M_mem = memory_bank.size()
    if M_mem == 0:
        return torch.zeros(d_h, device=device), None, [], []

    topk_lists = []
    for i_obs in known_attr:
        mem_obs = memory_bank.get_embeddings_for_attr(i_obs)
        if mem_obs is None or mem_obs.size(0) == 0:
            continue

        mem_obs = mem_obs.to(device)
        non_zero_mask = mem_obs.abs().sum(dim=1) > 1e-6
        if not non_zero_mask.any():
            continue

        query = F.normalize(known_embedding[i_obs], dim=0)
        mem_norm = F.normalize(mem_obs[non_zero_mask], dim=1)
        sim = torch.matmul(mem_norm, query)

        topk_local = torch.topk(sim, k=min(k, sim.numel())).indices.tolist()
        valid_positions = torch.where(non_zero_mask)[0].tolist()
        topk_positions = [valid_positions[j] for j in topk_local]
        topk_lists.append(topk_positions)

    candidate_positions = []
    if len(topk_lists) > 0:
        counter = Counter()
        for lst in topk_lists:
            counter.update(lst)

        threshold = min(r, len(topk_lists))
        candidate_positions = [p for p, cnt in counter.items() if cnt >= threshold]

        if len(candidate_positions) == 0 and len(counter) > 0:
            max_count = max(counter.values())
            candidate_positions = [p for p, cnt in counter.items() if cnt == max_count]

    mem_missing = memory_bank.get_embeddings_for_attr(missing_attr_idx)
    if mem_missing is None or mem_missing.size(0) == 0:
        return torch.zeros(d_h, device=device), None, [], []

    mem_missing = mem_missing.to(device)
    valid_mask_missing = memory_bank.get_valid_mask_for_attr(missing_attr_idx).to(device)
    non_zero_missing = mem_missing.abs().sum(dim=1) > 1e-6
    valid_mask_missing = valid_mask_missing & non_zero_missing

    candidate_positions = [p for p in candidate_positions if p < mem_missing.size(0)]
    reliable_positions = [p for p in candidate_positions if valid_mask_missing[p].item()]

    if len(reliable_positions) == 0:
        reliable_positions = torch.where(valid_mask_missing)[0].cpu().tolist()

    if len(reliable_positions) == 0:
        return torch.zeros(d_h, device=device), None, [], []

    selected_embeddings = mem_missing[reliable_positions]
    init_embedding = selected_embeddings.mean(dim=0)

    init_value = None
    values_missing = memory_bank.get_values_for_attr(missing_attr_idx)
    selected_values = [values_missing[p] for p in reliable_positions]

    if missing_modality == "num":
        valid_values = []
        for v in selected_values:
            try:
                fv = float(v)
                if fv != -1:
                    valid_values.append(fv)
            except:
                pass
        if len(valid_values) > 0:
            init_value = sum(valid_values) / len(valid_values)

    elif missing_modality == "cat":
        valid_values = [int(v) for v in selected_values if v is not None]
        if len(valid_values) > 0:
            init_value = Counter(valid_values).most_common(1)[0][0]

    elif missing_modality == "txt":
        init_value = None

    retrieved_event_row_ids = [memory_bank.get_row_ids()[p] for p in reliable_positions]
    return init_embedding, init_value, reliable_positions, retrieved_event_row_ids

def apply_memory_initialization(
    llm,
    rawn,
    cati,
    attr_mask,
    memory_bank,
    modality_types,
    k=5,
    r=2,
):
    llm = llm.clone()
    rawn = rawn.clone()
    cati = cati.clone()

    B, M0, D0 = llm.shape
    device = llm.device

    for b in range(B):
        known_attr = torch.where(attr_mask[b] == 1)[0].tolist()
        missing_attr = torch.where(attr_mask[b] == 0)[0].tolist()

        for miss_id in missing_attr:
            init_emb, init_val, _, _ = initialize_missing_embeddings_theory(
                missing_attr_idx=miss_id,
                known_attr=known_attr,
                known_embedding=llm[b],
                memory_bank=memory_bank,
                modality_types=modality_types,
                k=k,
                r=r,
                d_h=D0,
            )

            llm[b, miss_id, :] = init_emb.to(device)

            if modality_types[miss_id] == "cat" and init_val is not None:
                local_cat = miss_id - len(TEXTUAL_COLUMNS)
                cati[b, local_cat] = int(init_val)

            elif modality_types[miss_id] == "num" and init_val is not None:
                local_num = miss_id - len(TEXTUAL_COLUMNS) - len(CATEGORICAL_COLUMNS)
                rawn[b, local_num] = float(init_val)

    return llm, rawn, cati

# -----------------------------
# 17) MARIP LOSS
# -----------------------------
class MARIPLoss(nn.Module):
    def __init__(self, alpha=0.1, epsilon=1e-6):
        super().__init__()
        self.alpha = alpha
        self.epsilon = epsilon
        self.class_counts = {}
        self.observed_freq = {}
        self.num_scale = 0.5
        self.cat_scale = 0.5

    def register_class_counts(self, attr_idx, class_labels):
        class_labels = np.asarray(class_labels)
        unique, counts = np.unique(class_labels, return_counts=True)
        self.class_counts[attr_idx] = dict(zip(unique.tolist(), counts.tolist()))

    def register_observed_frequency(self, attr_idx, observed_count, total):
        self.observed_freq[attr_idx] = observed_count / (total + self.epsilon)

    def compute_lambda(self, idx, modality, y_true_value=None):
        obs = self.observed_freq.get(idx, 0.1)
        lam_miss = (1.0 / (obs + self.epsilon)) ** 0.5
        lam_miss = float(np.clip(lam_miss, 0.75, 2.5))

        lam_cls = 1.0
        lam_num = 1.0

        if modality == "cat":
            if idx in self.class_counts and y_true_value is not None:
                counts_dict = self.class_counts[idx]
                count = counts_dict.get(int(y_true_value), 1)
                total_count = sum(counts_dict.values())
                num_classes = max(len(counts_dict), 1)
                avg_count = total_count / num_classes
                lam_cls = 1.0 + (avg_count / (count + self.epsilon)) ** 0.5
                lam_cls = float(np.clip(lam_cls, 1.0, 3.0))
            lam = lam_miss * lam_cls
        elif modality == "num":
            lam = lam_miss * lam_num
        else:
            lam = 1.0

        lam = float(np.clip(lam, 1.0, 3.0))
        return lam

marip_loss_fn = MARIPLoss(alpha=0.1)

for j, col in enumerate(CATEGORICAL_COLUMNS):
    global_attr_idx = len(TEXTUAL_COLUMNS) + j
    marip_loss_fn.register_class_counts(global_attr_idx, ground_truth_cat[:, j])

total_n = len(df)
for j, col in enumerate(CATEGORICAL_COLUMNS):
    global_attr_idx = len(TEXTUAL_COLUMNS) + j
    observed_count = int(df[col].notna().sum())
    marip_loss_fn.register_observed_frequency(global_attr_idx, observed_count, total_n)

for j, col in enumerate(NUMERICAL_COLUMNS):
    global_attr_idx = len(TEXTUAL_COLUMNS) + len(CATEGORICAL_COLUMNS) + j
    observed_count = int(df[col].notna().sum())
    marip_loss_fn.register_observed_frequency(global_attr_idx, observed_count, total_n)

def compute_masked_marip_loss(cat_outs, num_outs, cat_targets, num_targets, attr_mask, marip_loss_fn, loss_on_masked_only=False):
    total_loss = 0.0
    valid_count = 0

    for j, logits in enumerate(cat_outs):
        global_attr_idx = len(TEXTUAL_COLUMNS) + j
        y_true = cat_targets[:, j].long().to(logits.device)

        masked_rows = (attr_mask[:, global_attr_idx] == 0)
        visible_rows = (attr_mask[:, global_attr_idx] == 1)
        use_rows = masked_rows if loss_on_masked_only else visible_rows

        if not use_rows.any():
            continue

        logits_use = logits[use_rows]
        y_true_use = y_true[use_rows]
        rec_loss = F.cross_entropy(logits_use, y_true_use, reduction="none")

        lam_list = []
        for b in range(y_true_use.size(0)):
            lam = marip_loss_fn.compute_lambda(global_attr_idx, "cat", y_true_use[b].item())
            lam_list.append(lam)

        lam_tensor = torch.tensor(lam_list, dtype=rec_loss.dtype, device=rec_loss.device)
        total_loss = total_loss + marip_loss_fn.cat_scale * (lam_tensor * rec_loss).mean()
        valid_count += 1

    for j, pred in enumerate(num_outs):
        global_attr_idx = len(TEXTUAL_COLUMNS) + len(CATEGORICAL_COLUMNS) + j
        y_true = num_targets[:, j].to(pred.device)
        valid_num = y_true != -1

        masked_rows = (attr_mask[:, global_attr_idx] == 0)
        visible_rows = (attr_mask[:, global_attr_idx] == 1)
        use_rows = masked_rows if loss_on_masked_only else visible_rows
        use_valid = use_rows & valid_num

        if not use_valid.any():
            continue

        pred_use = pred[use_valid]
        y_true_use = y_true[use_valid]
        rec_loss = F.mse_loss(pred_use, y_true_use, reduction="none")

        lam_list = []
        for b in range(y_true_use.size(0)):
            lam = marip_loss_fn.compute_lambda(global_attr_idx, "num", float(y_true_use[b].item()))
            lam_list.append(lam)

        lam_tensor = torch.tensor(lam_list, dtype=rec_loss.dtype, device=rec_loss.device)
        total_loss = total_loss + marip_loss_fn.num_scale * (lam_tensor * rec_loss).mean()
        valid_count += 1

    if valid_count == 0:
        return torch.tensor(0.0, device=attr_mask.device, requires_grad=True)

    return total_loss

# -----------------------------
# 18) MODEL
# -----------------------------
class VectorizedCrossModalGNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.Wg = nn.Linear(d, d, bias=False)
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wc = nn.Linear(d, d, bias=False)
        self.a = nn.Parameter(torch.randn(2 * d))

    def forward(self, Z, edge_index):
        B, M0, D0 = Z.shape
        if edge_index.numel() == 0:
            Q = self.Wq(Z)
            K = self.Wk(Z)
            beta = F.softmax(torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(D0), dim=-1)
            global_msg = torch.matmul(beta, self.Wc(Z))
            return F.leaky_relu(global_msg + Z)

        WgZ = self.Wg(Z)

        src = edge_index[0]
        tgt = edge_index[1]

        src_feat = WgZ[:, src, :]
        tgt_feat = WgZ[:, tgt, :]
        pair = torch.cat([src_feat, tgt_feat], dim=-1)
        e = F.leaky_relu(pair @ self.a)

        alpha = torch.zeros(B, M0, M0, device=Z.device)
        alpha[:, src, tgt] = e

        for i in range(M0):
            nbrs = tgt[src == i]
            if len(nbrs) > 0:
                alpha[:, i, nbrs] = F.softmax(alpha[:, i, nbrs], dim=-1)

        local_msg = torch.matmul(alpha, WgZ)

        Q = self.Wq(Z)
        K = self.Wk(Z)
        beta = F.softmax(torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(D0), dim=-1)
        global_msg = torch.matmul(beta, self.Wc(Z))

        return F.leaky_relu(local_msg + global_msg + Z)

class FastRAGNetNYCCrash(nn.Module):
    def __init__(self, embed_dim=384, d_shared=384, d_num=32, d_cat=16,
                 num_numerical_attrs=5, num_categorical_attrs=3,
                 num_classes_per_cat_attr=None, memory_bank=None):
        super().__init__()
        if num_classes_per_cat_attr is None:
            num_classes_per_cat_attr = [10] * num_categorical_attrs

        self.text_proj = nn.Linear(embed_dim, d_shared)

        self.num_proj = nn.ModuleList([
            nn.Sequential(nn.Linear(1, d_num), nn.ReLU(), nn.LayerNorm(d_num))
            for _ in range(num_numerical_attrs)
        ])
        self.cat_emb = nn.ModuleList([
            nn.Embedding(n_cls, d_cat) for n_cls in num_classes_per_cat_attr
        ])

        self.num_shared = nn.ModuleList([
            nn.Linear(embed_dim + d_num, d_shared) for _ in range(num_numerical_attrs)
        ])
        self.cat_shared = nn.ModuleList([
            nn.Linear(embed_dim + d_cat, d_shared) for _ in range(num_categorical_attrs)
        ])

        self.gnn = VectorizedCrossModalGNN(d_shared)
        self.memory_bank = memory_bank

        self.gate_h = nn.Linear(d_shared, d_shared)
        self.gate_m = nn.Linear(d_shared, d_shared)

        self.num_heads = nn.ModuleList([nn.Linear(d_shared, 1) for _ in range(num_numerical_attrs)])
        self.cat_heads = nn.ModuleList([nn.Linear(d_shared, n_cls) for n_cls in num_classes_per_cat_attr])

    def project_batch(self, llm_embeddings, raw_numericals, cat_indices):
        Z = []
        num_idx = 0
        cat_idx = 0

        for i, modality in enumerate(MODALITIES):
            x = llm_embeddings[:, i, :]

            if modality == "txt":
                z = self.text_proj(x)

            elif modality == "cat":
                c = self.cat_emb[cat_idx](cat_indices[:, cat_idx])
                z = self.cat_shared[cat_idx](torch.cat([x, c], dim=-1))
                cat_idx += 1

            else:
                n = raw_numericals[:, num_idx].unsqueeze(-1)
                nvec = self.num_proj[num_idx](n)
                z = self.num_shared[num_idx](torch.cat([x, nvec], dim=-1))
                num_idx += 1

            Z.append(z)

        return torch.stack(Z, dim=1)

    def memory_fusion(self, H, attr_mask=None):
        if self.memory_bank is None:
            return H

        out = H.clone()
        B, M0, D0 = H.shape

        for i in range(M0):
            mem = self.memory_bank.get_embeddings_for_attr(i)
            if mem is None:
                continue

            mem = mem.to(H.device)
            valid_mask = self.memory_bank.get_valid_mask_for_attr(i).to(H.device)
            non_zero_mask = mem.abs().sum(dim=1) > 1e-8
            use_mask = valid_mask & non_zero_mask

            if use_mask.sum() == 0:
                continue

            mem = mem[use_mask]
            mem = mem / (mem.norm(dim=1, keepdim=True) + 1e-8)

            h = H[:, i, :]
            h_norm = h / (h.norm(dim=1, keepdim=True) + 1e-8)

            sim = torch.matmul(h_norm, mem.T)
            attn = F.softmax(sim, dim=-1)
            r = torch.matmul(attn, mem)

            g = torch.sigmoid(self.gate_h(h) + self.gate_m(r))
            out[:, i, :] = g * r + (1 - g) * h

        return out

    def forward(self, llm_embeddings, raw_numericals, cat_indices, edge_index, attr_mask=None):
        Z = self.project_batch(llm_embeddings, raw_numericals, cat_indices)
        H = self.gnn(Z, edge_index)
        H = self.memory_fusion(H, attr_mask=attr_mask)

        cat_outs = []
        num_outs = []

        cat_idx = 0
        num_idx = 0
        for i, modality in enumerate(MODALITIES):
            if modality == "cat":
                cat_outs.append(self.cat_heads[cat_idx](H[:, i, :]))
                cat_idx += 1
            elif modality == "num":
                num_outs.append(self.num_heads[num_idx](H[:, i, :]).squeeze(-1))
                num_idx += 1

        return cat_outs, num_outs, H

model = FastRAGNetNYCCrash(
    embed_dim=384,
    d_shared=384,
    d_num=32,
    d_cat=16,
    num_numerical_attrs=len(NUMERICAL_COLUMNS),
    num_categorical_attrs=len(CATEGORICAL_COLUMNS),
    num_classes_per_cat_attr=num_classes_per_cat_attr,
    memory_bank=memory_bank
).to(DEVICE)

# -----------------------------
# 19) TRAIN TENSORS
# -----------------------------
X_all = normalized_embeddings

cat_targets_t = torch.tensor(ground_truth_cat, dtype=torch.long)
num_targets_t = torch.tensor(ground_truth_num, dtype=torch.float32)

raw_num_train = numerical_values_tensor[train_idx]
raw_num_test = numerical_values_tensor[test_idx]

cat_val_train = categorical_indices_tensor[train_idx]
cat_val_test = categorical_indices_tensor[test_idx]

# -----------------------------
# 20) TRAIN LOSS
# -----------------------------
def compute_masked_training_loss(cat_outs, num_outs, H, caty, numy, attr_mask,
                                 marip_loss_fn=None,
                                 loss_on_masked_only=False,
                                 reg_weight=1e-4,
                                 marip_blend=0.5,
                                 use_cat_class_consistency_loss=True,
                                 cat_class_consistency_weight=1e-2):
    recon_loss = 0.0
    valid_terms = 0

    cat_ptr = 0
    num_ptr = 0

    for attr_id, mod in enumerate(MODALITIES):
        masked_rows = (attr_mask[:, attr_id] == 0)
        visible_rows = (attr_mask[:, attr_id] == 1)
        use_rows = masked_rows if loss_on_masked_only else visible_rows

        if mod == "cat":
            if use_rows.any():
                logits = cat_outs[cat_ptr][use_rows]
                target = caty[use_rows, cat_ptr]
                recon_loss = recon_loss + F.cross_entropy(logits, target)
                valid_terms += 1
            cat_ptr += 1

        elif mod == "num":
            valid_num = numy[:, num_ptr] != -1
            use_valid = use_rows & valid_num
            if use_valid.any():
                pred = num_outs[num_ptr][use_valid]
                target = numy[use_valid, num_ptr]
                recon_loss = recon_loss + F.mse_loss(pred, target)
                valid_terms += 1
            num_ptr += 1

    if valid_terms == 0:
        recon_loss = torch.tensor(0.0, device=H.device, requires_grad=True)

    marip_loss = torch.tensor(0.0, device=H.device)
    if marip_loss_fn is not None:
        marip_loss = compute_masked_marip_loss(
            cat_outs=cat_outs,
            num_outs=num_outs,
            cat_targets=caty,
            num_targets=numy,
            attr_mask=attr_mask,
            marip_loss_fn=marip_loss_fn,
            loss_on_masked_only=loss_on_masked_only
        )

    reg_loss = hidden_regularization(H, attr_mask=attr_mask)

    cat_consistency_loss = torch.tensor(0.0, device=H.device)
    if use_cat_class_consistency_loss:
        cat_consistency_loss = categorical_class_consistency_loss(
            H=H,
            cat_targets=caty,
            attr_mask=attr_mask,
            visible_only=CONSISTENCY_ON_VISIBLE_ONLY
        )

    if marip_loss_fn is not None:
        total_loss = (
            (1.0 - marip_blend) * recon_loss
            + marip_blend * marip_loss
            + reg_weight * reg_loss
            + cat_class_consistency_weight * cat_consistency_loss
        )
    else:
        total_loss = recon_loss + reg_weight * reg_loss + cat_class_consistency_weight * cat_consistency_loss

    return (
        total_loss,
        recon_loss.detach(),
        marip_loss.detach(),
        reg_loss.detach(),
        cat_consistency_loss.detach()
    )

# -----------------------------
# 21) EVALUATION HELPERS
# -----------------------------
def retrieve_text_from_memory(query_h, mem_emb, mem_vals):
    query_h = query_h / (query_h.norm(dim=1, keepdim=True) + 1e-8)
    mem_emb = mem_emb / (mem_emb.norm(dim=1, keepdim=True) + 1e-8)
    sim = torch.matmul(query_h, mem_emb.T)
    idx = sim.argmax(dim=1).cpu().tolist()
    return [mem_vals[i] for i in idx]

def evaluate_standard(model, X_eval, raw_num_eval, cat_val_eval, cat_true, num_true):
    model.eval()

    cat_preds = []
    num_preds = []

    with torch.no_grad():
        for s in range(0, len(X_eval), 256):
            llm = X_eval[s:s+256].to(DEVICE)
            rawn = raw_num_eval[s:s+256].to(DEVICE)
            cati = cat_val_eval[s:s+256].to(DEVICE)

            cat_outs, num_outs, _ = model(llm, rawn, cati, edge_index, attr_mask=None)

            cur_cat = [x.argmax(dim=-1).cpu().numpy() for x in cat_outs]
            cur_num = [x.cpu().numpy() for x in num_outs]

            if len(cur_cat) > 0:
                cat_preds.append(np.stack(cur_cat, axis=1))
            if len(cur_num) > 0:
                num_preds.append(np.stack(cur_num, axis=1))

    cat_preds = np.concatenate(cat_preds, axis=0) if len(cat_preds) else np.zeros((len(X_eval), 0), dtype=np.int64)
    num_preds = np.concatenate(num_preds, axis=0) if len(num_preds) else np.zeros((len(X_eval), 0), dtype=np.float32)

    cat_true_np = cat_true.cpu().numpy()
    num_true_np = num_true.cpu().numpy()

    overall_cat_f1 = None
    overall_cat_acc = None
    if cat_true_np.size > 0:
        overall_cat_f1 = float(f1_score(cat_true_np.reshape(-1), cat_preds.reshape(-1), average="weighted", zero_division=0))
        overall_cat_acc = float((cat_true_np.reshape(-1) == cat_preds.reshape(-1)).mean())

    overall_num_mae = None
    overall_num_mse = None
    if num_true_np.size > 0:
        valid = num_true_np != -1
        if valid.any():
            overall_num_mae = float(np.mean(np.abs(num_true_np[valid] - num_preds[valid])))
            overall_num_mse = float(np.mean((num_true_np[valid] - num_preds[valid]) ** 2))

    cat_attr_metrics = compute_weighted_cat_f1_per_attribute(cat_true_np, cat_preds, CATEGORICAL_COLUMNS)
    num_attr_metrics = compute_weighted_num_mae_per_attribute(num_true_np, num_preds, NUMERICAL_COLUMNS)

    return {
        "cat_f1_weighted_global": overall_cat_f1,
        "cat_acc_weighted_global": overall_cat_acc,
        "num_mae_weighted_global": overall_num_mae,
        "num_mse_weighted_global": overall_num_mse,
        "cat_attribute_f1": cat_attr_metrics,
        "num_attribute_mae": num_attr_metrics,
        "text_metrics": {
            "text_bleu_mean": None,
            "note": "Standard evaluation has no text generation head; BLEU is reported in masked text retrieval evaluation."
        }
    }

def evaluate_single_masked_attribute(model, X_eval, raw_num_eval, cat_val_eval, cat_true, num_true, eval_df, attr_id, batch_size=256):
    model.eval()

    all_cat_preds = []
    all_num_preds = []
    text_preds = []

    with torch.no_grad():
        for s in range(0, len(X_eval), batch_size):
            llm = X_eval[s:s+batch_size].to(DEVICE).clone()
            rawn = raw_num_eval[s:s+batch_size].to(DEVICE).clone()
            cati = cat_val_eval[s:s+batch_size].to(DEVICE).clone()

            B = llm.size(0)
            attr_mask = torch.ones(B, len(MODALITIES), device=DEVICE)
            attr_mask[:, attr_id] = 0.0

            llm, rawn, cati = apply_input_mask(llm, rawn, cati, attr_mask)

            if USE_MISSING_INIT:
                llm, rawn, cati = apply_memory_initialization(
                    llm=llm,
                    rawn=rawn,
                    cati=cati,
                    attr_mask=attr_mask,
                    memory_bank=model.memory_bank,
                    modality_types=MODALITIES,
                    k=MISSING_INIT_TOPK,
                    r=MISSING_INIT_R,
                )

            cat_outs, num_outs, H = model(llm, rawn, cati, edge_index, attr_mask=attr_mask)

            cur_cat = [x.argmax(dim=-1).cpu().numpy() for x in cat_outs]
            cur_num = [x.cpu().numpy() for x in num_outs]

            if len(cur_cat) > 0:
                all_cat_preds.append(np.stack(cur_cat, axis=1))
            if len(cur_num) > 0:
                all_num_preds.append(np.stack(cur_num, axis=1))

            if MODALITIES[attr_id] == "txt":
                mem_emb = model.memory_bank.get_embeddings_for_attr(attr_id)
                mem_vals = model.memory_bank.get_values_for_attr(attr_id)
                valid_mask = model.memory_bank.get_valid_mask_for_attr(attr_id)

                if mem_emb is not None and mem_vals is not None and len(mem_vals) > 0:
                    mem_emb = mem_emb[valid_mask]
                    mem_vals = [v for j, v in enumerate(mem_vals) if valid_mask[j].item()]
                    if len(mem_vals) > 0:
                        preds = retrieve_text_from_memory(H[:, attr_id, :], mem_emb.to(DEVICE), mem_vals)
                    else:
                        preds = [""] * B
                else:
                    preds = [""] * B

                text_preds.extend(preds)

    cat_preds = np.concatenate(all_cat_preds, axis=0) if len(all_cat_preds) else None
    num_preds = np.concatenate(all_num_preds, axis=0) if len(all_num_preds) else None

    if MODALITIES[attr_id] == "cat":
        local_cat = attr_id - len(TEXTUAL_COLUMNS)
        y_true = cat_true.cpu().numpy()[:, local_cat]
        y_pred = cat_preds[:, local_cat]

        evaluated_count = int(len(y_true))
        f1_val = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))

        return {
            "attr_id": int(attr_id),
            "attr_name": CATEGORICAL_COLUMNS[local_cat],
            "attr_type": "cat",
            "weighted_f1": f1_val,
            "accuracy": float((y_true == y_pred).mean()),
            "evaluated_masked_count": evaluated_count
        }

    elif MODALITIES[attr_id] == "num":
        local_num = attr_id - len(TEXTUAL_COLUMNS) - len(CATEGORICAL_COLUMNS)
        y_true = num_true.cpu().numpy()[:, local_num]
        y_pred = num_preds[:, local_num]

        valid = y_true != -1
        valid_count = int(valid.sum())

        mae = np.mean(np.abs(y_true[valid] - y_pred[valid])) if valid.any() else None
        mse = np.mean((y_true[valid] - y_pred[valid]) ** 2) if valid.any() else None

        return {
            "attr_id": int(attr_id),
            "attr_name": NUMERICAL_COLUMNS[local_num],
            "attr_type": "num",
            "weighted_mae": float(mae) if mae is not None else None,
            "weighted_mse": float(mse) if mse is not None else None,
            "evaluated_masked_count": valid_count
        }

    else:
        local_txt = attr_id
        refs = eval_df[TEXTUAL_COLUMNS[local_txt]].fillna("").astype(str).tolist()
        bleu = compute_text_bleu(refs, text_preds)
        return {
            "attr_id": int(attr_id),
            "attr_name": TEXTUAL_COLUMNS[local_txt],
            "attr_type": "txt",
            "text_bleu_mean": float(bleu),
            "evaluated_masked_count": int(len(refs))
        }

def evaluate_full(model, eval_idx):
    X_eval = X_all[eval_idx]
    raw_num_eval = numerical_values_tensor[eval_idx]
    cat_val_eval = categorical_indices_tensor[eval_idx]
    cat_true = cat_targets_t[eval_idx]
    num_true = num_targets_t[eval_idx]
    eval_df = df.iloc[eval_idx].reset_index(drop=True)

    standard_results = evaluate_standard(
        model=model,
        X_eval=X_eval,
        raw_num_eval=raw_num_eval,
        cat_val_eval=cat_val_eval,
        cat_true=cat_true,
        num_true=num_true
    )

    masked_results = []
    for attr_id in range(len(MODALITIES)):
        res = evaluate_single_masked_attribute(
            model=model,
            X_eval=X_eval,
            raw_num_eval=raw_num_eval,
            cat_val_eval=cat_val_eval,
            cat_true=cat_true,
            num_true=num_true,
            eval_df=eval_df,
            attr_id=attr_id,
            batch_size=256
        )
        masked_results.append(res)

    cat_items = [
        x for x in masked_results
        if x["attr_type"] == "cat"
        and x.get("weighted_f1") is not None
        and x.get("evaluated_masked_count") is not None
    ]
    num_items = [
        x for x in masked_results
        if x["attr_type"] == "num"
        and x.get("weighted_mae") is not None
        and x.get("evaluated_masked_count") is not None
    ]
    txt_items = [
        x for x in masked_results
        if x["attr_type"] == "txt"
        and x.get("text_bleu_mean") is not None
        and x.get("evaluated_masked_count") is not None
    ]

    masked_weighted_attribute_f1, total_cat_weight = weighted_average_by_count(
        cat_items, "weighted_f1", "evaluated_masked_count"
    )
    masked_weighted_attribute_mae, total_num_weight = weighted_average_by_count(
        num_items, "weighted_mae", "evaluated_masked_count"
    )
    masked_weighted_attribute_mse, _ = weighted_average_by_count(
        num_items, "weighted_mse", "evaluated_masked_count"
    )
    masked_text_bleu_mean, total_txt_weight = weighted_average_by_count(
        txt_items, "text_bleu_mean", "evaluated_masked_count"
    )

    out = dict(standard_results)
    out["masked_attribute_results"] = masked_results
    out["masked_summary"] = {
        "masked_weighted_attribute_f1": masked_weighted_attribute_f1,
        "masked_weighted_attribute_f1_total_weight": total_cat_weight,
        "masked_weighted_attribute_mae": masked_weighted_attribute_mae,
        "masked_weighted_attribute_mae_total_weight": total_num_weight,
        "masked_weighted_attribute_mse": masked_weighted_attribute_mse,
        "masked_text_bleu_mean": masked_text_bleu_mean,
        "masked_text_bleu_total_weight": total_txt_weight
    }
    out["eval_loss"] = None
    return out

# -----------------------------
# 22) TRAINING
# -----------------------------
def train_ragnet(model, train_idx, test_idx, epochs=3, lr=1e-4, eval_every=1):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    best_score = -1e18
    best_path = os.path.join(MODEL_DIR, "nyc_crash_ragnet_best.pt")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        total_recon_loss = 0.0
        total_marip_loss = 0.0
        total_reg_loss = 0.0
        total_cat_consistency_loss = 0.0
        valid_train = 0

        shuffled = np.random.permutation(train_idx)

        for s in tqdm(range(0, len(shuffled), TRAIN_BATCH_SIZE), desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            batch_ids = shuffled[s:s+TRAIN_BATCH_SIZE]

            llm = X_all[batch_ids].to(DEVICE).clone()
            rawn = numerical_values_tensor[batch_ids].to(DEVICE).clone()
            cati = categorical_indices_tensor[batch_ids].to(DEVICE).clone()
            caty = cat_targets_t[batch_ids].to(DEVICE)
            numy = num_targets_t[batch_ids].to(DEVICE)

            if USE_RANDOM_MASK_TRAINING:
                attr_mask = build_training_attr_mask(
                    batch_size=llm.size(0),
                    num_attrs=len(MODALITIES),
                    device=DEVICE,
                    mask_one_attr_per_sample=MASK_ONE_ATTR_PER_SAMPLE,
                    mask_prob=MASK_PROB
                )

                llm, rawn, cati = apply_input_mask(llm, rawn, cati, attr_mask)

                if USE_MISSING_INIT:
                    llm, rawn, cati = apply_memory_initialization(
                        llm=llm,
                        rawn=rawn,
                        cati=cati,
                        attr_mask=attr_mask,
                        memory_bank=memory_bank,
                        modality_types=MODALITIES,
                        k=MISSING_INIT_TOPK,
                        r=MISSING_INIT_R,
                    )
            else:
                attr_mask = torch.ones(llm.size(0), len(MODALITIES), device=DEVICE)

            cat_outs, num_outs, H = model(llm, rawn, cati, edge_index, attr_mask=attr_mask)

            loss, recon_loss, marip_loss, reg_loss, cat_consistency_loss = compute_masked_training_loss(
                cat_outs=cat_outs,
                num_outs=num_outs,
                H=H,
                caty=caty,
                numy=numy,
                attr_mask=attr_mask,
                marip_loss_fn=marip_loss_fn if USE_MARIP else None,
                loss_on_masked_only=LOSS_ON_MASKED_ONLY,
                reg_weight=REG_WEIGHT,
                marip_blend=MARIP_BLEND,
                use_cat_class_consistency_loss=USE_CAT_CLASS_CONSISTENCY_LOSS,
                cat_class_consistency_weight=CAT_CLASS_CONSISTENCY_WEIGHT
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += float(loss.item())
            total_recon_loss += float(recon_loss.item())
            total_marip_loss += float(marip_loss.item())
            total_reg_loss += float(reg_loss.item())
            total_cat_consistency_loss += float(cat_consistency_loss.item())
            valid_train += 1

        avg_train_loss = total_train_loss / max(valid_train, 1)
        avg_recon_loss = total_recon_loss / max(valid_train, 1)
        avg_marip_loss = total_marip_loss / max(valid_train, 1)
        avg_reg_loss = total_reg_loss / max(valid_train, 1)
        avg_cat_consistency_loss = total_cat_consistency_loss / max(valid_train, 1)

        if (epoch + 1) % eval_every == 0:
            eval_results = evaluate_full(model, test_idx)
        else:
            eval_results = {"eval_loss": None, "masked_summary": {}}

        row = {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_recon_loss": avg_recon_loss,
            "train_marip_loss": avg_marip_loss,
            "train_reg_loss": avg_reg_loss,
            "train_cat_consistency_loss": avg_cat_consistency_loss,
            "cat_f1_weighted_global": eval_results.get("cat_f1_weighted_global"),
            "num_mae_weighted_global": eval_results.get("num_mae_weighted_global"),
            "masked_weighted_attribute_f1": eval_results.get("masked_summary", {}).get("masked_weighted_attribute_f1"),
            "masked_weighted_attribute_mae": eval_results.get("masked_summary", {}).get("masked_weighted_attribute_mae"),
            "masked_text_bleu_mean": eval_results.get("masked_summary", {}).get("masked_text_bleu_mean"),
        }
        history.append(row)

        print(f"\n[Epoch {epoch+1}/{epochs}]")
        print(f"Train Loss                     : {avg_train_loss:.4f}")
        print(f"Train Recon Loss               : {avg_recon_loss:.4f}")
        print(f"Train MARIP Loss               : {avg_marip_loss:.4f}")
        print(f"Train Reg Loss                 : {avg_reg_loss:.6f}")
        print(f"Train Cat Consistency Loss     : {avg_cat_consistency_loss:.6f}")
        print(f"Cat F1 Weighted Global         : {eval_results.get('cat_f1_weighted_global', 0.0) or 0.0:.4f}")
        print(f"Num MAE Weighted Global        : {eval_results.get('num_mae_weighted_global', 0.0) or 0.0:.4f}")
        print(f"Masked Weighted Attribute F1   : {eval_results.get('masked_summary', {}).get('masked_weighted_attribute_f1', 0.0) or 0.0:.4f}")
        print(f"Masked Weighted Attribute MAE  : {eval_results.get('masked_summary', {}).get('masked_weighted_attribute_mae', 0.0) or 0.0:.4f}")
        print(f"Masked Text BLEU Mean          : {eval_results.get('masked_summary', {}).get('masked_text_bleu_mean', 0.0) or 0.0:.4f}")

        cur_score = 0.0
        if eval_results.get("masked_summary", {}).get("masked_weighted_attribute_f1") is not None:
            cur_score += float(eval_results["masked_summary"]["masked_weighted_attribute_f1"])
        if eval_results.get("masked_summary", {}).get("masked_text_bleu_mean") is not None:
            cur_score += float(eval_results["masked_summary"]["masked_text_bleu_mean"])
        if eval_results.get("masked_summary", {}).get("masked_weighted_attribute_mae") is not None:
            cur_score -= float(eval_results["masked_summary"]["masked_weighted_attribute_mae"])

        if cur_score > best_score:
            best_score = cur_score
            torch.save({"model_state_dict": model.state_dict(), "history": history}, best_path)

    return history

history = train_ragnet(model, train_idx, test_idx, epochs=EPOCHS, lr=LR, eval_every=1)

# -----------------------------
# 23) FINAL EVALUATION
# -----------------------------
final_results = evaluate_full(model, test_idx)

print("\n===== FINAL TEST RESULTS =====")
for k, v in final_results.items():
    if isinstance(v, dict):
        print(f"{k}: {v}")
    elif isinstance(v, list):
        print(f"{k}: {v[:3]} ... total={len(v)}")
    else:
        print(f"{k}: {v:.4f}" if isinstance(v, (int, float, np.floating)) and v is not None else f"{k}: {v}")

# -----------------------------
# 24) SAVE OUTPUTS
# -----------------------------
np.save(os.path.join(EMB_DIR, "textual_embeddings.npy"), textual_embeddings)
np.save(os.path.join(EMB_DIR, "categorical_embeddings.npy"), categorical_embeddings)
np.save(os.path.join(EMB_DIR, "numerical_embeddings.npy"), numerical_embeddings)
np.save(os.path.join(EMB_DIR, "categorical_indices.npy"), categorical_indices)
np.save(os.path.join(EMB_DIR, "numerical_values.npy"), numerical_values)

np.save(os.path.join(OUT_DIR, "all_embeddings.npy"), all_embeddings_np)
np.save(os.path.join(OUT_DIR, "normalized_embeddings.npy"), normalized_embeddings.detach().cpu().numpy())
np.save(os.path.join(OUT_DIR, "ground_truth_cat.npy"), ground_truth_cat)
np.save(os.path.join(OUT_DIR, "ground_truth_num.npy"), ground_truth_num)
np.save(os.path.join(OUT_DIR, "target_labels.npy"), target_labels)

torch.save(edge_index.detach().cpu(), os.path.join(OUT_DIR, "edge_index.pt"))
torch.save({"model_state_dict": model.state_dict()}, os.path.join(MODEL_DIR, "nyc_crash_ragnet_last.pt"))

with open(os.path.join(OUT_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

with open(os.path.join(OUT_DIR, "final_results.json"), "w") as f:
    json.dump(final_results, f, indent=2, default=float)

metadata = {
    "raw_data_path": RAW_DATA_PATH,
    "save_dir": SAVE_DIR,
    "textual_columns": TEXTUAL_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numerical_columns": NUMERICAL_COLUMNS,
    "target_column": TARGET_COLUMN,
    "modalities": MODALITIES,
    "all_attribute_names": ALL_ATTRIBUTE_NAMES,
    "num_classes_per_cat_attr": num_classes_per_cat_attr,
    "target_map": target_map,
    "train_subset": len(train_idx),
    "test_subset": len(test_idx),
    "epochs": EPOCHS,
    "variant": "nyc-crash-raw-preprocess-corrected-event-memory-balanced-same-class-embedding-missing-init-weighted-masked-metrics",
    "memory_size": memory_bank.size(),
    "memory_row_ids": memory_bank.get_row_ids(),
    "use_missing_init": USE_MISSING_INIT,
    "missing_init_topk": MISSING_INIT_TOPK,
    "missing_init_r": MISSING_INIT_R,
    "max_rows_after_preprocess": MAX_ROWS_AFTER_PREPROCESS
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved files to:", SAVE_DIR)
print("Done.")

DEVICE: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found CSV files: ['/content/drive/MyDrive/NYC_crash/h9gi-nx95.csv']
Loaded /content/drive/MyDrive/NYC_crash/h9gi-nx95.csv: shape=(1000, 30)
Combined raw shape: (1000, 30)
Preprocessed shape: (1000, 9)
                                          crash_text   borough  \
0  Crash in missing on whitestone expressway near...   Missing   
1  Crash in missing on queensboro bridge upper du...   Missing   
2  Crash in brooklyn on ocean parkway near avenue...  BROOKLYN   

            on_street_name contributing_factor_vehicle_1  latitude  longitude  \
0    WHITESTONE EXPRESSWAY  Aggressive Driving/Road Rage       NaN        NaN   
1  QUEENSBORO BRIDGE UPPER             Pavement Slippery       NaN        NaN   
2            OCEAN PARKWAY                   Unspecified  40.62179 -73.970024   

   number_of_persons_injured  number_of_persons_killed  hour_of_day

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
  0%|          | 0/8 [00:00<?, ?it/s]/tmp/ipykernel_7191/2277450617.py:482: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
  0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_7191/2277450617.py:482: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
  0%|          | 0/4 [00:00<?, ?it/s]/tmp/ipykernel_7191/2277450617.py:482: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)`

Event-based memory bank built. size = 300
First 10 memory row ids: [438, 872, 965, 287, 831, 514, 633, 795, 993, 665]
edge_index: torch.Size([2, 18])



[Epoch 1/100]
Train Loss                     : 14.5550
Train Recon Loss               : 14.8319
Train MARIP Loss               : 12.0620
Train Reg Loss                 : 0.053981
Train Cat Consistency Loss     : 0.007141
Cat F1 Weighted Global         : 0.4153
Num MAE Weighted Global        : 0.3036
Masked Weighted Attribute F1   : 0.2079
Masked Weighted Attribute MAE  : 0.4051
Masked Text BLEU Mean          : 0.3773



[Epoch 2/100]
Train Loss                     : 13.3657
Train Recon Loss               : 13.5861
Train MARIP Loss               : 11.3816
Train Reg Loss                 : 0.056538
Train Cat Consistency Loss     : 0.006790
Cat F1 Weighted Global         : 0.5722
Num MAE Weighted Global        : 0.2145
Masked Weighted Attribute F1   : 0.2708
Masked Weighted Attribute MAE  : 0.4001
Masked Text BLEU Mean          : 0.3769


...


[Epoch 96/100]
Train Loss                     : 0.2370
Train Recon Loss               : 0.2259
Train MARIP Loss               : 0.2577
Train Reg Loss                 : 2.890687
Train Cat Consistency Loss     : 0.769350
Cat F1 Weighted Global         : 0.8307
Num MAE Weighted Global        : 0.0380
Masked Weighted Attribute F1   : 0.3449
Masked Weighted Attribute MAE  : 0.3960
Masked Text BLEU Mean          : 0.4222



[Epoch 97/100]
Train Loss                     : 0.2166
Train Recon Loss               : 0.2057
Train MARIP Loss               : 0.2306
Train Reg Loss                 : 2.886723
Train Cat Consistency Loss     : 0.814476
Cat F1 Weighted Global         : 0.8265
Num MAE Weighted Global        : 0.0379
Masked Weighted Attribute F1   : 0.3446
Masked Weighted Attribute MAE  : 0.3957
Masked Text BLEU Mean          : 0.4225



[Epoch 98/100]
Train Loss                     : 0.2679
Train Recon Loss               : 0.2556
Train MARIP Loss               : 0.2972
Train Reg Loss                 : 3.009388
Train Cat Consistency Loss     : 0.780757
Cat F1 Weighted Global         : 0.8232
Num MAE Weighted Global        : 0.0383
Masked Weighted Attribute F1   : 0.3464
Masked Weighted Attribute MAE  : 0.3954
Masked Text BLEU Mean          : 0.4216



[Epoch 99/100]
Train Loss                     : 0.2134
Train Recon Loss               : 0.2021
Train MARIP Loss               : 0.2293
Train Reg Loss                 : 3.059450
Train Cat Consistency Loss     : 0.822486
Cat F1 Weighted Global         : 0.8273
Num MAE Weighted Global        : 0.0384
Masked Weighted Attribute F1   : 0.3449
Masked Weighted Attribute MAE  : 0.3954
Masked Text BLEU Mean          : 0.4206



[Epoch 100/100]
Train Loss                     : 0.2212
Train Recon Loss               : 0.2097
Train MARIP Loss               : 0.2415
Train Reg Loss                 : 3.046593
Train Cat Consistency Loss     : 0.799013
Cat F1 Weighted Global         : 0.8267
Num MAE Weighted Global        : 0.0376
Masked Weighted Attribute F1   : 0.3452
Masked Weighted Attribute MAE  : 0.3962
Masked Text BLEU Mean          : 0.4213

===== FINAL TEST RESULTS =====
cat_f1_weighted_global: 0.8267
cat_acc_weighted_global: 0.8317
num_mae_weighted_global: 0.0376
num_mse_weighted_global: 0.0039
cat_attribute_f1: {'borough': 1.0, 'on_street_name': 0.4936666666666666, 'contributing_factor_vehicle_1': 0.9883333333333334, 'weighted_attribute_f1': 0.8273333333333333}
num_attribute_mae: {'latitude': 0.047438088804483414, 'longitude': 0.04544927179813385, 'number_of_persons_injured': 0.0286589153110981, 'number_of_persons_killed': 0.025726214051246643, 'hour_of_day': 0.04213092848658562, 'weighted_attribute_mae': 